

# 08. Dwellsy Data Exploration

## 1. Introduction

### Objective

This notebook presents an initial inspection of the real rental listing data provided by Dwellsy for the Accolade Capstone Project.

The goal is to identify the dataset structure, target variable, potential predictors, and key data quality considerations needed to develop preliminary rental price prediction models.

A comprehensive exploratory data analysis and feature selection process is being conducted separately by other team members.

### Data sources

The analysis uses historical rental listing data for:

- Chicago Metropolitan Statistical Area (MSA).
- Washington, D.C. Metropolitan Statistical Area (MSA).

The datasets are provided as compressed CSV files, accompanied by the `TotalIQ Overview v5.xlsx` data documentation.
  

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


## 2. Data Loading and Initial Inspection

### 2.1 File Locations and Formats

The Dwellsy datasets are stored locally as compressed CSV files.

To minimize memory usage, we initially load only five observations from each dataset. This allows us to inspect their structure without loading the full datasets into memory.

These initial records are used for structural inspection only and are not intended to be representative samples.
  


In [2]:
# Define the project root
project_root = Path.cwd().resolve()

# If the notebook is running from its own directory,
# navigate to the project root
if project_root.name == "phase_01_market_rent":
    project_root = project_root.parent.parent

# Define the directory containing the Dwellsy files
data_dir = project_root / "data" / "dwellsy" / "Real data"

# Define file paths
chicago_path = data_dir / "Dwellsy_Trial Data_Chicago MSA_Full HIstory.csv.zip"
dc_path = data_dir / "Dwellsy_Trial Data_DC MSA_Full HIstory.csv.zip"

# Check that both files exist
print("Chicago file exists:", chicago_path.exists())
print("DC file exists:", dc_path.exists())

# Read only the first five rows of each dataset
chicago_sample = pd.read_csv(chicago_path, nrows=5, low_memory=False)
dc_sample = pd.read_csv(dc_path, nrows=5, low_memory=False)

# Display dataset dimensions for the loaded samples
print("\nChicago sample shape:", chicago_sample.shape)
print("DC sample shape:", dc_sample.shape)

# Display the first five records from Chicago
display(chicago_sample.head())

Chicago file exists: True
DC file exists: True

Chicago sample shape: (5, 47)
DC sample shape: (5, 47)


,dwellsy_address_id,address1_id,address2_id,data_source_external_id,listing_id,community_id,community_name,company_name,top_down_community_count,top_down_address1_count,...,ss_raw_secondary_designator,ss_raw_secondary_number,ss_raw_default_city_name,ss_raw_state_abbreviation,ss_raw_zipcode,ss_raw_plus4_code,amenities,photos,uru_id,uru_type
0,133822-8163780,133822,8163780,1739819,3675156,279577,6211 N Kenmore Ave,Mihail Marinov,28,28,...,Apt,501,Chicago,IL,60660,2151,12 Month Lease or Longer; Cable Ready; Cats OK...,https://media.dwellsy.com/prod/c151dd5f3919432...,542230,Apartment
1,129521-8112041,129521,8112041,3871623,3558998,6069979,Mode Logan Square,Jessica Goodnight,296,218,...,Apt,807,Chicago,IL,60642,6024,12 Month Lease or Longer; Air Conditioning; Ca...,https://media.dwellsy.com/prod/8c186d97caa2102...,1362919,Apartment
2,123813-8066896,123813,8066896,14582015,16668829,279329,21st Street Lofts,3L Living,137,137,...,Unit,433,Chicago,IL,60616,1732,Bike Storage; Cats OK; Deck/Porch/Patio; Eleva...,https://media.dwellsy.com/prod/9f1aec6e910f4f3...,5009610,Apartment
3,123813-8066896,123813,8066896,14582015,19878436,279329,21st Street Lofts,3L Living,137,137,...,Unit,433,Chicago,IL,60616,1732,Bike Storage; Cats OK; Deck/Porch/Patio; Eleva...,https://media.dwellsy.com/prod/9f1aec6e910f4f3...,5009610,Apartment
4,130630-8121250,130630,8121250,10139825,10639335,5764,2420 N Kedzie Blvd,Sam,54,54,...,Apt,206,Chicago,IL,60647,5740,Cats OK; Dishwasher; Gas included in rent; Har...,https://media.dwellsy.com/prod/cd5a80b0a1d4a17...,428523,Apartment



### 2.2 Initial Variable Inspection

Before developing preliminary rental price prediction models, we examine the variables available in the Chicago and Washington, D.C. datasets.

The purpose of this step is to identify the target variable, potential predictors, listing and property identifiers, and temporal information required for model development.

This is a targeted inspection to support preliminary modeling, rather than a comprehensive exploratory data analysis.
  

In [4]:
# Compare variables across datasets
chicago_columns = set(chicago_sample.columns)
dc_columns = set(dc_sample.columns)

print("Number of columns in Chicago:", len(chicago_columns))
print("Number of columns in DC:", len(dc_columns))

print("\nColumns shared by both datasets:")
print(sorted(chicago_columns & dc_columns))

print("\nColumns only in Chicago:")
print(sorted(chicago_columns - dc_columns))

print("\nColumns only in DC:")
print(sorted(dc_columns - chicago_columns))

Number of columns in Chicago: 47
Number of columns in DC: 47

Columns shared by both datasets:
['address1_id', 'address2_id', 'address_1', 'address_2', 'address_city', 'address_country', 'address_state', 'address_type', 'address_zip', 'amenities', 'bedrooms', 'community_id', 'community_name', 'company_name', 'creation_time', 'data_source_external_id', 'deactivation_time', 'description', 'dwellsy_address_id', 'full_baths', 'half_baths', 'latitude', 'listing_deposit', 'listing_id', 'longitude', 'msa_code', 'msa_name', 'photos', 'property_listing_status', 'rent_amount', 'square_feet', 'ss_raw_default_city_name', 'ss_raw_plus4_code', 'ss_raw_primary_number', 'ss_raw_secondary_designator', 'ss_raw_secondary_number', 'ss_raw_state_abbreviation', 'ss_raw_street_name', 'ss_raw_street_postdirection', 'ss_raw_street_predirection', 'ss_raw_street_suffix', 'ss_raw_zipcode', 'top_down_address1_count', 'top_down_community_count', 'uru_id', 'uru_type', 'year_built']

Columns only in Chicago:
[]

Colu


### 2.3 Schema Comparison and Preliminary Feature Identification

The Chicago and Washington, D.C. datasets contain the same 47 variables, with no city-specific differences in their column names.

This consistent schema allows us to develop a common preprocessing and modeling pipeline for both metropolitan areas.

The preliminary target variable is `rent_amount`, representing the rental amount associated with a listing.

Potential predictors include property characteristics (`bedrooms`, `full_baths`, `half_baths`, `square_feet`, `year_built`), location information (`address_zip`, `msa_name`), and property attributes (`amenities`, `uru_type`).

The datasets also include listing and property identifiers (`listing_id`, `uru_id`) and temporal variables (`creation_time`, `deactivation_time`). These fields will be important for investigating repeated observations and designing an appropriate train-test split.

Feature selection and preprocessing decisions remain preliminary and will be refined as the project progresses.



### 2.4 Preliminary Modeling Variables

To support the development of initial rental price prediction models, we examine a small subset of variables related to rental prices, property characteristics, location, and record identification.

This inspection focuses on data types and example values. Missingness rates and other data quality statistics will be evaluated on a larger sample before model training.


In [5]:
modeling_columns = [
    "rent_amount",
    "bedrooms",
    "full_baths",
    "half_baths",
    "square_feet",
    "year_built",
    "address_zip",
    "uru_type",
    "uru_id",
    "listing_id",
    "creation_time"
]

# Inspect preliminary modeling variables
print("Chicago:")
display(chicago_sample[modeling_columns])

print("\nData types:")
print(chicago_sample[modeling_columns].dtypes)

Chicago:


,rent_amount,bedrooms,full_baths,half_baths,square_feet,year_built,address_zip,uru_type,uru_id,listing_id,creation_time
0,1145.0,1,1,NaN,800,NaN,60660,Apartment,542230,3675156,2021-02-28 15:00:43
1,2500.0,1,1,NaN,909,NaN,60642,Apartment,1362919,3558998,2021-02-14 07:11:47
2,1699.0,1,1,NaN,572,2017.0,60616,Apartment,5009610,16668829,2025-05-15 13:35:48
3,1705.0,1,1,NaN,572,2017.0,60616,Apartment,5009610,19878436,2026-05-18 15:15:54
4,1450.0,0,1,NaN,400,1929.0,60647,Apartment,428523,10639335,2023-06-21 01:50:14



Data types:
rent_amount      float64
bedrooms           int64
full_baths         int64
half_baths       float64
square_feet        int64
year_built       float64
address_zip        int64
uru_type             str
uru_id             int64
listing_id         int64
creation_time        str
dtype: object



### 2.5 Preliminary Data Structure Findings

The initial inspection of five Chicago observations reveals several considerations for preliminary model development:

- **Target variable:** `rent_amount` is available as a numeric variable.
- **Numerical predictors:** `bedrooms`, `full_baths`, and `square_feet` are read as numeric variables.
- **Missing values:** Some property characteristics, including `half_baths` and `year_built`, contain missing values.
- **Categorical variables:** `address_zip` should be treated as a categorical geographic identifier rather than a continuous numerical variable.
- **Temporal information:** `creation_time` is initially read as a string and requires datetime conversion.
- **Repeated units:** The same `uru_id` appears in multiple records with different `listing_id` values, rental amounts, and creation dates.

These findings suggest that preliminary modeling should include appropriate missing-value handling, categorical encoding, and a train-test splitting strategy that accounts for repeated rental units.

The observations above are based on a five-row structural sample. Their prevalence in the full dataset has not yet been established.



### 2.6 Dataset Dimensions

Before selecting a working sample for preliminary modeling, we determine the total number of observations in each dataset.

Both datasets contain 47 variables. Since the CSV files include text fields that may contain embedded line breaks, counting physical lines could overestimate the number of observations.

To obtain accurate record counts, we use pandas to read each compressed CSV in chunks, counting the parsed records without loading the full datasets into memory.

This allows us to determine the actual dataset dimensions and select an appropriate sample size for preliminary model training.

In [9]:
# Count actual CSV records without loading the full datasets

def count_csv_records(zip_path):
    total_records = 0

    for chunk in pd.read_csv(
        zip_path,
        usecols=["listing_id"],
        chunksize=100_000
    ):
        total_records += len(chunk)

    return total_records


# Count observations in both datasets
n_chicago = count_csv_records(chicago_path)
n_dc = count_csv_records(dc_path)

# Display dataset dimensions
print(f"Chicago: {n_chicago:,} observations, 47 variables")
print(f"Washington, D.C.: {n_dc:,} observations, 47 variables")
print(f"Total: {n_chicago + n_dc:,} observations")

Chicago: 545,312 observations, 47 variables
Washington, D.C.: 430,605 observations, 47 variables
Total: 975,917 observations




### Dataset Size: Initial Findings

The Dwellsy datasets contain 975,917 historical rental listing records across two metropolitan areas:

| Metropolitan Area | Observations | Variables |
|---|---:|---:|
| Chicago MSA | 545,312 | 47 |
| Washington, D.C. MSA | 430,605 | 47 |
| **Total** | **975,917** | **47** |

Chicago accounts for approximately 55.9% of the records, while Washington, D.C. accounts for 44.1%.

Record counts were obtained using pandas to correctly parse the CSV files, accounting for potential embedded line breaks in text fields.

Importantly, the number of records does not necessarily correspond to the number of unique rental units, since the datasets contain historical listings and the same unit may appear multiple times.

These datasets provide a substantial foundation for developing and evaluating rental price prediction models.






## 3. Preliminary Modeling Dataset

### 3.1 Random Sampling of Chicago Rental Listings

To develop preliminary rental price prediction models, we construct a reproducible random sample of approximately 100,000 records from the Chicago dataset.

The full Chicago dataset contains 545,312 historical rental listing records. Although the dataset is manageable, using a smaller sample allows us to develop and evaluate preliminary models more efficiently.

We use chunk-based processing to read the compressed CSV incrementally, retaining a randomly selected subset of observations. The sampling fraction is approximately 18.34%.

Only variables relevant to preliminary modeling and record identification are loaded.

The sample will be used to assess data quality, develop preprocessing procedures, and test preliminary modeling pipelines before training models on the complete Chicago dataset.

**Limitation:** Sampling individual records does not eliminate repeated observations of the same rental unit. This issue will be addressed when constructing the training and test sets.



In [10]:
# Candidate variables for preliminary modeling
selected_columns = [
    "rent_amount",
    "bedrooms",
    "full_baths",
    "half_baths",
    "square_feet",
    "year_built",
    "address_zip",
    "uru_type",
    "uru_id",
    "listing_id",
    "creation_time"
]

# Sampling parameters
seed = 826
target_sample_size = 100_000
sampling_fraction = target_sample_size / n_chicago

rng = np.random.default_rng(seed)
sampled_chunks = []

# Read Chicago data in chunks
for chunk in pd.read_csv(
    chicago_path,
    usecols=selected_columns,
    chunksize=100_000
):
    # Randomly select observations from each chunk
    mask = rng.random(len(chunk)) < sampling_fraction

    sampled_chunk = chunk.loc[mask]

    if not sampled_chunk.empty:
        sampled_chunks.append(sampled_chunk)

# Combine sampled observations
chicago_df = pd.concat(sampled_chunks, ignore_index=True)

# Display results
print("Original dataset size:", f"{n_chicago:,}")
print("Sample shape:", chicago_df.shape)
print("Sampling fraction:", round(sampling_fraction * 100, 2), "%")

Original dataset size: 545,312
Sample shape: (99979, 11)
Sampling fraction: 18.34 %



### 3.2 Missing Values Assessment

Before training preliminary rental price prediction models, we examine missing values in the selected variables.

Missing values in the target variable (`rent_amount`) must be handled before model training, while missing values in predictor variables may require imputation or other preprocessing strategies.

We calculate the number and percentage of missing observations for each variable to guide subsequent data preparation decisions.


In [11]:
# Calculate missing values for each variable
missing_summary = pd.DataFrame({
    "Missing count": chicago_df.isna().sum(),
    "Missing percentage": chicago_df.isna().mean() * 100
})

# Display missing values summary
display(missing_summary.round(2))

,Missing count,Missing percentage
listing_id,0,0.00
address_zip,0,0.00
year_built,32512,32.52
bedrooms,0,0.00
full_baths,25,0.03
half_baths,82553,82.57
square_feet,5390,5.39
rent_amount,0,0.00
creation_time,0,0.00
uru_id,0,0.00



### 3.3 Preliminary Feature Selection and Missing Data Strategy

The missing value assessment reveals that the target variable (`rent_amount`) is complete across all 99,979 sampled observations.

Most candidate predictors have relatively low missingness. In particular, `square_feet` has 5.39% missing values, while `full_baths` has only 0.03%.

Two variables require special consideration:

- `half_baths` has 82.57% missing values and will be excluded from the initial models.
- `year_built` has 32.52% missing values but will be retained initially, using median imputation.

For the preliminary modeling pipeline, we select the following predictors:

- **Numerical:** `bedrooms`, `full_baths`, `square_feet`, and `year_built`.
- **Categorical:** `address_zip` and `uru_type`.

Numerical missing values will be imputed using medians estimated from the training data.

The identifiers `listing_id` and `uru_id` will not be used as predictors. However, `uru_id` will be retained to support a group-aware train-test split, reducing the risk of having records from the same rental unit in both sets.

These decisions are preliminary and may be revised after evaluating model performance and data quality.



### 3.4 Rental Price Distribution and Initial Data Quality Checks

Before proceeding with model development, we examine the distribution of the target variable (`rent_amount`) and perform basic data quality checks.

We calculate descriptive statistics, including selected percentiles, to identify potentially unusual rental prices and assess the presence of extreme values.

We also check for non-positive rental prices and square footage values, which could indicate invalid observations.

These checks will help determine whether additional filtering is necessary before model training.


In [12]:
# Summary statistics for rental prices
print("Rental price summary:")
display(
    chicago_df["rent_amount"].describe(
        percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
    )
)

# Check non-positive rental prices
print(
    "Non-positive rents:",
    (chicago_df["rent_amount"] <= 0).sum()
)

# Check missing or non-positive square footage
print(
    "Non-positive square footage:",
    (chicago_df["square_feet"] <= 0).sum()
)

Rental price summary:


count    99979.000000
mean      1914.169970
std        978.711989
min        267.000000
1%         766.000000
5%         900.000000
50%       1695.000000
95%       3600.000000
99%       5415.000000
max      20000.000000
Name: rent_amount, dtype: float64

Non-positive rents: 0
Non-positive square footage: 0




**Initial Findings**

The rental price distribution is right-skewed, with a mean monthly rent of approximately $1,914 and a median of $1,695.

The 1st and 99th percentiles are $766 and $5,415, respectively, while the maximum observed rent is $20,000.

No non-positive rental prices or square footage values were identified in the sample.

Although some rental prices are relatively high, these observations are not necessarily erroneous. Therefore, no upper-bound filtering is applied at this stage.

Potential outliers and their influence on predictive performance may be investigated in subsequent modeling experiments.





### 3.5 Saving the Preliminary Modeling Sample

To ensure reproducibility and avoid repeatedly processing the original compressed CSV files, we save the sampled Chicago dataset as an intermediate dataset.

The resulting file contains 99,979 observations and 11 selected variables.

This sample can be reused for exploratory analysis, preprocessing tests, and rapid model prototyping.

For the main preliminary modeling experiments, we plan to use the complete Chicago dataset (545,312 observations), subject to basic data quality checks and appropriate preprocessing.



In [13]:
# Define output directory
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# Save preliminary modeling dataset
sample_path = output_dir / "chicago_modeling_sample.csv"

chicago_df.to_csv(sample_path, index=False)

print("Sample saved to:", sample_path)
print("Dataset shape:", chicago_df.shape)

Sample saved to: /Users/maggonllama/Documents/Maestría/Columbia University/Semestre 3/Capstone Project/Accolade/From-Listings-to-Intelligence/data/processed/chicago_modeling_sample.csv
Dataset shape: (99979, 11)
